In [ ]:
import os
import sys
import json
from pathlib import Path


import pandas as pd
import numpy as np
import altair as alt


import geopandas as gpd
from shapely.geometry import Point

pd.set_option('display.max_rows', None)

pd.set_option('display.max_columns', None)
pd.set_option("display.width", 160)

print("Versions ->",
      "pandas:", pd.__version__,
      "| geopandas:", gpd.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
df_sub = pd.read_pickle("/content/drive/MyDrive/424-Assignment-3/df_sub.pkl")

print("Shape:", df_sub.shape)
df_sub.info()

# Display first few rows.
df_sub.head(5)

In [ ]:
# Build df_v1 from df_sub
import pandas as pd

if 'df_sub' not in globals():
    df_sub = pd.read_pickle("/content/drive/MyDrive/424-Assignment-3/df_sub.pkl")

df_v1 = (
    df_sub[
        ["Borough", "Job Type", "Initial Cost", "Total Construction Floor Area", "Filing Date"]
    ].copy()
    .rename(columns={
        "Job Type": "JobType",
        "Initial Cost": "InitialCost",
        "Total Construction Floor Area": "FloorArea",
        "Filing Date": "FilingDate"
    })
)

# Clean types & rows
df_v1["InitialCost"] = pd.to_numeric(df_v1["InitialCost"], errors="coerce")
df_v1["FloorArea"]   = pd.to_numeric(df_v1["FloorArea"], errors="coerce")
df_v1 = df_v1.dropna(subset=["Borough", "JobType", "InitialCost", "FloorArea"])

print("df_v1 shape:", df_v1.shape)
df_v1.head(3)

# Linked View Visualization 2: Initial Cost vs Floor Area with Borough Filtering

In [ ]:
import numpy as np
import altair as alt

alt.data_transformers.enable('default', max_rows=None)
alt.renderers.enable('colab')

if 'df_linked' not in locals():
    df_linked = df_v1[
        (df_v1["InitialCost"] > 100) & (df_v1["InitialCost"] < 1e7) &
        (df_v1["FloorArea"]  > 100)  & (df_v1["FloorArea"]  < 1e5)
    ].sample(800, random_state=42).copy()

# Log features
df_linked["logCost"] = np.log10(df_linked["InitialCost"])
df_linked["logArea"] = np.log10(df_linked["FloorArea"])

# Mid-range domains
q = df_linked[["logArea", "logCost"]].quantile([0.05, 0.95])
x_min, x_max = float(q.loc[0.05, "logArea"]), float(q.loc[0.95, "logArea"])
y_min, y_max = float(q.loc[0.05, "logCost"]), float(q.loc[0.95, "logCost"])
pad_x = 0.10 * (x_max - x_min)
pad_y = 0.10 * (y_max - y_min)
x_dom = [x_min - pad_x, x_max + pad_x]
y_dom = [y_min - pad_y, y_max + pad_y]

# Extra padding so marks don’t touch the y-axis
left_pad = 0.15 * (x_dom[1] - x_dom[0])
x_left   = x_dom[0] + left_pad

# Selections
brush  = alt.selection_interval(encodings=["x", "y"], translate=True, zoom=True)
jobSel = alt.selection_point(fields=["JobType"], bind="legend", empty="all", toggle=True)


# Scatter (Job Type legend only)
scatter = (
    alt.Chart(df_linked)
      .transform_filter(alt.datum.logArea > x_left)
      .mark_circle(size=68, opacity=0.8, stroke='white', strokeWidth=0.5)
      .encode(
          x=alt.X("logArea:Q",
                  title="log₁₀ (Floor Area)",
                  scale=alt.Scale(domain=[x_left, x_dom[1]], clamp=True, nice=False),
                  axis=alt.Axis(labelFlush=False)),
          y=alt.Y("logCost:Q",
                  title="log₁₀ (Initial Cost)",
                  scale=alt.Scale(domain=y_dom, clamp=True, nice=False)),
          color=alt.Color("JobType:N", legend=alt.Legend(title="Job Type")),
          opacity=alt.condition(jobSel, alt.value(1.0), alt.value(0.15)),
          tooltip=["Borough", "JobType", "InitialCost", "FloorArea"]
      )
      .add_params(brush, jobSel)
      .properties(width=650, height=360,
                  title="Initial Cost vs Floor Area (log₁₀) — brush to filter below")
)


#Histogram (Borough legend only)
hist = (
    alt.Chart(df_linked)
      .transform_filter(jobSel)
      .transform_filter(brush)
      .mark_bar()
      .encode(
          x=alt.X("Borough:N", title="Borough"),
          y=alt.Y("count():Q", title="Number of Projects"),
          color=alt.Color("Borough:N", legend=alt.Legend(title="Borough")),
          tooltip=[alt.Tooltip("Borough:N", title="Borough"),
                   alt.Tooltip("count():Q", title="Projects Count")]
      )
      .properties(width=650, height=200,
                  title="Projects per Borough (filtered by brush & Job Type legend)")
)

# Keep legends/scales separate across the two views
linked_view = (
    alt.vconcat(scatter, hist)
      .resolve_scale(color='independent')
      .configure_view(stroke=None)
)

linked_view